In [ ]:
"""
SVM from scratch in PyTorch — hinge loss formulation.

Objective (what you derived today):
    minimize  (1/2)||w||^2  +  C * sum_i max(0, 1 - y_i * (w^T x_i + b))

This is the UNCONSTRAINED equivalent of the primal QP. We optimize it
directly with gradient descent instead of solving via Lagrangian/dual —
that's what makes this "from scratch" rather than calling sklearn.svm.SVC
(which solves the dual QP under the hood).

Fill in every TODO yourself. Don't peek at a library implementation until
you've gotten this training and separating the data correctly.
"""

import torch

torch.manual_seed(0)

# ---------------------------------------------------------------------------
# 1. Toy dataset — linearly separable, two clusters, labels in {-1, +1}
#    (not {0,1} — you derived why earlier today).
# ---------------------------------------------------------------------------
n_per_class = 50
X_pos = torch.randn(n_per_class, 2) + torch.tensor([2.0, 2.0])
X_neg = torch.randn(n_per_class, 2) + torch.tensor([-2.0, -2.0])
X = torch.cat([X_pos, X_neg], dim=0)
y = torch.cat([torch.ones(n_per_class), -torch.ones(n_per_class)])  # {+1, -1}

# ---------------------------------------------------------------------------
# 2. Parameters to learn: w (2,) and b (scalar). Must track gradients.
# ---------------------------------------------------------------------------
w = torch.randn(2, requires_grad=True)  # TODO: is zero-init reasonable here? think about it
b = torch.zeros(1, requires_grad=True)

C = 1.0   # tradeoff: margin width vs. violation penalty
lr = 0.01
n_epochs = 200

# ---------------------------------------------------------------------------
# 3. Hinge loss function.
#    L(w,b) = 0.5 * ||w||^2 + C * mean( max(0, 1 - y*(X@w + b)) )
#    TODO: implement. Use torch.clamp(..., min=0) for max(0, ...).
# ---------------------------------------------------------------------------
def hinge_loss(X, y, w, b, C):
    z = X @ w + b       # TODO: f(x) = X @ w + b            shape (n,)
    margins = y * z     # TODO: margins = y * f(x)
    per_point_loss = torch.clamp(1-margins, 0)    # TODO: per_point_loss = max(0, 1 - margins)

    total_loss = 0.5 * (torch.norm(w)**2) + C * torch.mean(per_point_loss)        # TODO: combine with 0.5 * ||w||^2
    return total_loss
    raise NotImplementedError("fill this in")


# ---------------------------------------------------------------------------
# 4. Training loop — plain gradient descent, no torch.optim, so you see
#    exactly what's updating and why.
# ---------------------------------------------------------------------------
for epoch in range(n_epochs):
    loss = hinge_loss(X, y, w, b, C)

    # TODO: zero out w.grad / b.grad before backward()
    loss.backward()    # TODO: loss.backward()
    
    # TODO: manually update w, b using .grad inside torch.no_grad():
    #       w -= lr * w.grad ; b -= lr * b.grad
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()


    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

# ---------------------------------------------------------------------------
# 5. Sanity check after training:
#    predictions = sign(X @ w + b), compare to y, print accuracy.
scores = X @ w + b

# Convert scores to class predictions (-1 or +1)
predictions = torch.where(scores >= 0, 1, -1)

# Compute accuracy
accuracy = (predictions == y).float().mean()

print(f"Accuracy: {accuracy.item() * 100:.2f}%")
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 6. REFLECTION (answer once training works — don't skip):
#
#    a) Which points end up with hinge loss > 0 after convergence? Print
#       per-point loss and check — these are your support vectors in the
#       soft-margin sense.
#
#    b) What happens to the decision boundary if C is very small (0.01)
#       vs very large (100)? Compare resulting w and margin width (2/||w||).
#
#    c) This dataset is linearly separable — no kernel needed. If you
#       wanted this exact code to handle XOR-shaped data, what's the ONE
#       thing you'd change? (Hint: Block 2 — training loop doesn't change.)
# ---------------------------------------------------------------------------

tensor([2.8969, 4.0882, 2.8331, 3.5239, 3.5471, 3.2926, 4.3840, 3.7808, 3.8262,
        2.7880, 3.9866, 2.9781, 4.6726, 4.2011, 2.4998, 2.6304, 3.6456, 3.8630,
        3.0528, 2.8522, 3.7265, 2.4313, 4.3153, 2.9187, 3.7606, 2.6788, 4.8078,
        5.1503, 2.5441, 4.3708, 2.6513, 2.6477, 2.0016, 3.0110, 3.0630, 2.2554,
        2.9180, 3.2424, 3.4729, 3.6687, 4.0220, 3.3937, 2.6999, 2.1217, 4.1701,
        3.9356, 3.8189, 3.0379, 2.1689, 2.6446, 2.5055, 2.3653, 3.2742, 3.5089,
        2.7620, 4.4114, 2.8553, 3.3859, 4.0640, 3.9355, 2.9981, 2.5478, 2.5764,
        2.0789, 2.3421, 3.4926, 3.5046, 2.8994, 3.0489, 4.6880, 2.0865, 2.5299,
        2.4376, 4.4567, 2.3213, 2.9849, 3.4846, 2.6902, 3.7554, 4.8153, 2.9518,
        2.1172, 3.2214], grad_fn=<IndexBackward0>)
epoch   0  loss 3.5466
tensor([2.8138, 3.9500, 2.7867, 3.4415, 3.4313, 3.1642, 4.2575, 3.6486, 3.7418,
        2.6601, 3.8777, 2.8800, 4.5002, 4.0919, 2.4176, 2.5402, 3.5313, 3.7360,
        2.9400, 2.7602, 3.5643, 2.3334, 4.1818